In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: optional
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available(): raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# LoRALinear bằng PyTorch — không cần PEFT

PEFT là appendix online, không nằm trong profile PTIT 2026 của cẩm nang và notebook không tự cài package.

In [ ]:
class LoRALinear(torch.nn.Module):
    def __init__(self,base,rank=2,alpha=4):
        super().__init__(); self.base=base; self.scale=alpha/rank
        for p in base.parameters(): p.requires_grad=False
        self.A=torch.nn.Parameter(torch.randn(rank,base.in_features)*.01); self.B=torch.nn.Parameter(torch.zeros(base.out_features,rank))
    def forward(self,x): return self.base(x)+self.scale*(x@self.A.T)@self.B.T
base=torch.nn.Linear(8,6); layer=LoRALinear(base); x=torch.randn(4,8)
assert torch.allclose(layer(x),base(x)); trainable=sum(p.numel() for p in layer.parameters() if p.requires_grad)
assert trainable==2*(8+6); print("trainable",trainable)